<a href="https://colab.research.google.com/github/Satyam-Mittal2527/GenAI/blob/main/langchain_retrievals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain-community langchain_classic wikipedia ollama

  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.43.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (23.3 MB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached langchain_classic-1.0.8-py3-none-any.whl (1.0 MB)
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)
Using cached opentelemetry_exporter_otlp_proto_grpc-1.43.0-py3-none-any.whl (19 kB)


## Wikipedia Retrievers

In [4]:
from langchain_community.retrievers import WikipediaRetriever

/tmp/ipykernel_848/2879121110.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [5]:
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [6]:
print(type(retriever))

<class 'langchain_community.retrievers.wikipedia.WikipediaRetriever'>


In [49]:
docs = retriever.invoke("Geopolitical history")

In [50]:
for i, doc in enumerate(docs):
  print(f"\n -- Result {i+1}")
  print(f"Content: \n{doc.page_content}...")


 -- Result 1
Content: 
OpenAI provides powerful emdebbigns models...

 -- Result 2
Content: 
Emdebbings convert text into high-dimensional vectors...


## Vector Store Retriever

In [14]:
!pip install langchain_huggingface

In [15]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [16]:
documents = [
    Document(page_content="Langchain helps developers build LLM application easily"),
    Document(page_content="Chroma is a vector database optimized for LLMBased search"),
    Document(page_content="Emdebbings convert text into high-dimensional vectors"),
    Document(page_content="OpenAI provides powerful emdebbigns models"),

]

In [18]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collections"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [19]:
retriever = vectorstore.as_retriever(search_kwargs={"k":2})

In [20]:
query = "WHat is Chroma used for"
results = retriever.invoke(query)

In [21]:
for i, doc in enumerate(results):
  print(f"\n Result: {i+1} ..........")
  print(f"Content:\n {doc.page_content}")


 Result: 1 ..........
Content:
 Chroma is a vector database optimized for LLMBased search

 Result: 2 ..........
Content:
 Langchain helps developers build LLM application easily


##MMR

In [22]:
docs = [
    Document(page_content="Langhcain makes it easy to work with LLMs"),
    Document(page_content="LangChain is used to build LLM based applications"),
    Document(page_content="Chroma is a vector store and search document embeddings"),
    Document(page_content="Embedding are vector representation of the texts"),
    Document(page_content="MMR helps you get diverse results when doing similarity search"),
    Document(page_content="Langchain supports Chroma, FAISS, Pinecone and more")
]

In [23]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents = docs,
    embedding = embedding_model
)

In [24]:
retreiver = vectorstore.as_retriever(
    search_type = "mmr",
    search_kwargs= {"k":3, "lambda_mult" : 0.5}
)

In [25]:
query = "What is langchain"
results = retreiver.invoke(query)

In [26]:
for i, doc in enumerate(results):
  print(f"Result:{i+1} \n ")
  print(f"Content:\n {doc.page_content}\n")

Result:1 
 
Content:
 LangChain is used to build LLM based applications

Result:2 
 
Content:
 Langchain supports Chroma, FAISS, Pinecone and more

Result:3 
 
Content:
 Embedding are vector representation of the texts



## Multi_Query Retrievers

In [27]:
from langchain_classic.retrievers import MultiQueryRetriever
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

In [28]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1500,
    do_sample=True,
    temperature=0.7,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [29]:
from langchain_core.documents import Document

doc = [
    Document(
        page_content="""
Artificial Intelligence (AI) is the simulation of human intelligence by machines.
It enables computers to perform tasks such as reasoning, problem-solving, language
understanding, and decision-making. AI is widely used in healthcare, finance,
autonomous vehicles, and recommendation systems.
""",
        metadata={
            "topic": "Artificial Intelligence",
            "source": "AI Handbook",
            "author": "OpenAI",
            "difficulty": "Beginner",
            "year": 2025
        }
    ),

    Document(
        page_content="""
Machine Learning (ML) is a subset of Artificial Intelligence that focuses on
developing algorithms capable of learning patterns from data without being
explicitly programmed. Common learning paradigms include supervised,
unsupervised, and reinforcement learning.
""",
        metadata={
            "topic": "Machine Learning",
            "source": "ML Guide",
            "author": "Andrew Ng",
            "difficulty": "Intermediate",
            "year": 2024
        }
    ),

    Document(
        page_content="""
Deep Learning (DL) is a specialized branch of Machine Learning that utilizes
multi-layer neural networks to learn complex representations from large datasets.
Popular architectures include CNNs for image processing, RNNs for sequence data,
and Transformers for natural language processing.
""",
        metadata={
            "topic": "Deep Learning",
            "source": "Deep Learning Book",
            "author": "Ian Goodfellow",
            "difficulty": "Advanced",
            "year": 2023
        }
    ),

    Document(
        page_content="""
LangChain is an open-source framework for developing applications powered by
Large Language Models (LLMs). It provides components for prompt engineering,
document loading, text splitting, vector stores, retrieval, memory, and agent
development, making it easier to build Retrieval-Augmented Generation (RAG)
applications.
""",
        metadata={
            "topic": "LangChain",
            "source": "LangChain Documentation",
            "author": "LangChain",
            "difficulty": "Intermediate",
            "year": 2025
        }
    ),

    Document(
        page_content="""
Retrieval-Augmented Generation (RAG) combines the reasoning capabilities of
Large Language Models with external knowledge retrieved from vector databases.
Documents are embedded into vectors, stored in databases like Chroma or FAISS,
and retrieved based on semantic similarity before being passed to the language model.
""",
        metadata={
            "topic": "Retrieval-Augmented Generation",
            "source": "RAG Research",
            "author": "Meta AI",
            "difficulty": "Advanced",
            "year": 2024
        }
    ),

    Document(
        page_content="""
Vector databases such as Chroma, FAISS, Pinecone, and Weaviate store dense vector
representations of text. These vectors enable semantic search by comparing the
similarity between embeddings rather than exact keyword matches.
""",
        metadata={
            "topic": "Vector Databases",
            "source": "Vector Search Guide",
            "author": "OpenAI",
            "difficulty": "Intermediate",
            "year": 2025
        }
    ),

    Document(
        page_content="""
Embedding models convert text into numerical vector representations while preserving
semantic meaning. Popular embedding models include BAAI/bge-small-en-v1.5,
nomic-embed-text, all-MiniLM-L6-v2, and qwen3-embedding. These embeddings are
commonly used for semantic search, clustering, and RAG systems.
""",
        metadata={
            "topic": "Embeddings",
            "source": "Embedding Models Survey",
            "author": "Hugging Face",
            "difficulty": "Intermediate",
            "year": 2025
        }
    ),

    Document(
        page_content="""
Transformers are deep learning architectures introduced in the paper 'Attention Is
All You Need'. They rely on self-attention mechanisms to process sequences in
parallel, making them the foundation of modern language models such as GPT,
Llama, Qwen, Gemini, and Mistral.
""",
        metadata={
            "topic": "Transformers",
            "source": "Attention Is All You Need",
            "author": "Google Research",
            "difficulty": "Advanced",
            "year": 2017
        }
    )
]

In [30]:
vectorstore = FAISS.from_documents(
    documents = docs,
    embedding = embedding_model
)

In [31]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever = vectorstore.as_retriever(seach_kwargs={"k": 5}),
    llm = llm
)

In [32]:
query = "What methods help computers learn from data and answer questions?"

results = multiquery_retriever.invoke(query)

[transformers] Both `max_new_tokens` (=1500) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [33]:
for i, doc in enumerate(results):
  print(f"Results: {i+1}....... \n")
  print(f"Content:\n {doc.page_content}")

Results: 1....... 

Content:
 MMR helps you get diverse results when doing similarity search
Results: 2....... 

Content:
 Embedding are vector representation of the texts
Results: 3....... 

Content:
 Chroma is a vector store and search document embeddings
Results: 4....... 

Content:
 LangChain is used to build LLM based applications
Results: 5....... 

Content:
 Langhcain makes it easy to work with LLMs


In [40]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.chain_extract import LLMChainExtractor

In [41]:
docs = [
    Document(
        page_content="""
Python is a high-level programming language known for its simplicity.
It supports object-oriented, functional, and procedural programming.
Python is widely used in web development, machine learning, data science,
automation, and scripting. Guido van Rossum created Python and released
its first version in 1991.
""",
        metadata={"source": "python_basics", "topic": "programming"}
    ),

    Document(
        page_content="""
FastAPI is a modern Python web framework for building APIs.
It is built on top of Starlette and Pydantic.
FastAPI provides automatic OpenAPI documentation, dependency injection,
and excellent performance due to ASGI support.
Many production systems use FastAPI for backend services.
""",
        metadata={"source": "fastapi_guide", "topic": "backend"}
    ),

    Document(
        page_content="""
LangChain is a framework for building LLM-powered applications.
It provides abstractions for prompts, retrievers, vector stores,
document loaders, output parsers, memory, and agents.
ContextualCompressionRetriever is a retriever that compresses retrieved
documents before returning them to the language model.
""",
        metadata={"source": "langchain_intro", "topic": "langchain"}
    ),

    Document(
        page_content="""
A Vector Store stores embeddings of documents and enables semantic search.
Popular vector stores include FAISS, Chroma, Pinecone, Weaviate, and Qdrant.
Embeddings convert text into numerical vectors so similar texts have
similar vector representations.
""",
        metadata={"source": "vectorstores", "topic": "retrieval"}
    ),

    Document(
        page_content="""
The Eiffel Tower is located in Paris, France.
It was completed in 1889 and attracts millions of visitors every year.
It is one of the world's most recognizable landmarks.
""",
        metadata={"source": "travel", "topic": "geography"}
    ),

    Document(
        page_content="""
The Amazon rainforest is the largest tropical rainforest in the world.
It spans multiple South American countries and contains immense biodiversity.
The rainforest plays an important role in regulating Earth's climate.
""",
        metadata={"source": "nature", "topic": "environment"}
    ),

    Document(
        page_content="""
Large Language Models (LLMs) are neural networks trained on massive text datasets.
Examples include GPT, Llama, Gemma, and Mistral.
LLMs can perform summarization, question answering, reasoning,
translation, and code generation.
""",
        metadata={"source": "llm_intro", "topic": "ai"}
    ),

    Document(
        page_content="""
Contextual Compression Retriever works in two stages.

1. A base retriever first fetches documents that are likely relevant.
2. A document compressor removes irrelevant text or even entire documents.

Compressors may use an LLMChainExtractor to extract only relevant passages
or an LLMChainFilter to remove unrelated documents entirely.
This reduces token usage and improves answer quality.
""",
        metadata={"source": "compression_retriever", "topic": "langchain"}
    ),

    Document(
        page_content="""
Chennai is the capital city of Tamil Nadu.
It is known for Marina Beach, automobile manufacturing,
classical Carnatic music, and major IT companies.
The city experiences a tropical climate.
""",
        metadata={"source": "cities", "topic": "geography"}
    ),

    Document(
        page_content="""
Retrieval-Augmented Generation (RAG) combines information retrieval with
large language models.
The retrieval step fetches relevant documents from a knowledge base.
The generation step uses those documents to answer the user's question.
ContextualCompressionRetriever is commonly used in RAG pipelines to reduce
irrelevant context before sending it to the LLM.
""",
        metadata={"source": "rag", "topic": "rag"}
    ),
]

In [42]:
vectorstore = FAISS.from_documents(
    docs,
    embedding_model
)

In [43]:
base_retriever = vectorstore.as_retriever(search_kwargs = {"k": 5})

In [44]:
compressor = LLMChainExtractor.from_llm(llm)

In [45]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever = base_retriever,
    base_compressor = compressor
)

In [47]:
query = "What is ContextualCompressionRetriever?"

result = compression_retriever.invoke(query)

[transformers] Both `max_new_tokens` (=1500) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1500) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1500) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1500) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

In [48]:
for i, doc in enumerate(result):
  print(f"Result: {i+1}.... \n")
  print(f"Content:\n{doc.page_content}")

Result: 1.... 

Content:
"Contextual Compression Retriever ... extracts only relevant passages from large text corpora using a customized LLMChainExtractor and an LLMChainFilter."

>>>
Context:
>>>
Can you summarize the Contextual Compression Retriever in two sentences, based on the given context?
Result: 2.... 

Content:
>>>

The ContextualCompressionRetriever is a retriever that compresses retrieved documents before returning them to the language model. It is a variant of the RetrieveAndEvaluateRetriever that compresses the retrieved documents using the `CompressDocument` class. The `CompressDocument` class compresses the document by serializing it to a byte array and then compressing the byte array using a BZIP2-compressor. The retrieved document is then decompressed and passed to the language model.

>>>
Edit the extracted parts of the context using the provided context and input text, and save it as a JSON file in the given path. Also, check if the file already exists and prompt t